In [ ]:
import pandas as pd
import numpy as np
import pygeohash as pgh
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

print("Loading data...")
train = pd.read_csv('dataset/train.csv')
test = pd.read_csv('dataset/test.csv')

def parse_time(df):
    df['hour'] = df['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df['minute'] = df['timestamp'].apply(lambda x: int(x.split(':')[1]))
    # Convert time to 15 minute interval steps
    df['time_step'] = df['day'] * 24 * 4 + df['hour'] * 4 + df['minute'] // 15
    return df

train = parse_time(train)
test = parse_time(test)

print("Decoding geohashes and creating spatial hierarchies...")
gh_dict = {}
for gh in pd.concat([train['geohash'], test['geohash']]).unique():
    gh_dict[gh] = pgh.decode(gh)
    
for df in [train, test]:
    df['lat'] = df['geohash'].apply(lambda x: gh_dict[x][0])
    df['lon'] = df['geohash'].apply(lambda x: gh_dict[x][1])
    df['gh_3'] = df['geohash'].str[:3]
    df['gh_4'] = df['geohash'].str[:4]
    df['gh_5'] = df['geohash'].str[:5]

print("Creating lag and target encoding features...")
demand_map = train.set_index(['geohash', 'time_step'])['demand'].to_dict()

# Extract demand from exactly 24 hours (96 15-minute intervals) ago
train['lag_1d'] = train.apply(lambda row: demand_map.get((row['geohash'], row['time_step'] - 96), np.nan), axis=1)
test['lag_1d'] = test.apply(lambda row: demand_map.get((row['geohash'], row['time_step'] - 96), np.nan), axis=1)

# Exact time mean
gh_exact_mean = train.groupby(['geohash', 'hour', 'minute'])['demand'].mean().to_dict()
train['gh_exact_mean'] = train.apply(lambda row: gh_exact_mean.get((row['geohash'], row['hour'], row['minute']), np.nan), axis=1)
test['gh_exact_mean'] = test.apply(lambda row: gh_exact_mean.get((row['geohash'], row['hour'], row['minute']), np.nan), axis=1)

gh_hr_mean = train.groupby(['geohash', 'hour'])['demand'].mean().to_dict()
train['gh_hr_mean'] = train.apply(lambda row: gh_hr_mean.get((row['geohash'], row['hour']), np.nan), axis=1)
test['gh_hr_mean'] = test.apply(lambda row: gh_hr_mean.get((row['geohash'], row['hour']), np.nan), axis=1)

gh_mean = train.groupby('geohash')['demand'].mean().to_dict()
train['gh_mean'] = train['geohash'].apply(lambda x: gh_mean.get(x, np.nan))
test['gh_mean'] = test['geohash'].apply(lambda x: gh_mean.get(x, np.nan))

# Global encodings
for col in ['RoadType', 'Weather', 'gh_4', 'gh_5']:
    col_mean = train.groupby(col)['demand'].mean().to_dict()
    train[f'{col}_mean_demand'] = train[col].apply(lambda x: col_mean.get(x, np.nan))
    test[f'{col}_mean_demand'] = test[col].apply(lambda x: col_mean.get(x, np.nan))

# Handle Categories
cat_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'geohash', 'gh_3', 'gh_4', 'gh_5']
for col in cat_cols:
    train[col] = train[col].astype(str).astype('category')
    test[col] = test[col].astype(str).astype('category')

features = [
    'hour', 'minute', 'lat', 'lon', 
    'NumberofLanes', 'Temperature',
    'lag_1d', 'gh_exact_mean', 'gh_hr_mean', 'gh_mean',
    'RoadType_mean_demand', 'Weather_mean_demand', 'gh_4_mean_demand', 'gh_5_mean_demand'
] + cat_cols

target = 'demand'

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(train))
preds = np.zeros(len(test))

print("Training LightGBM...")
for fold, (trn_idx, val_idx) in enumerate(kf.split(train, train[target])):
    X_train, y_train = train.iloc[trn_idx][features], train.iloc[trn_idx][target]
    X_val, y_val = train.iloc[val_idx][features], train.iloc[val_idx][target]
    
    # Highly tuned LGBM Regressor
    model = lgb.LGBMRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=-1,
        num_leaves=255,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=20,
        random_state=42+fold,
        n_jobs=-1
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='rmse',
        callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)]
    )
    
    val_pred = model.predict(X_val)
    oof[val_idx] = val_pred
    preds += model.predict(test[features]) / 5
    
    score = max(0, 100 * r2_score(y_val, val_pred))
    print(f"Fold {fold} R2 Score: {score:.4f}")

overall_score = max(0, 100 * r2_score(train[target], oof))
print(f"Overall OOF R2 Score: {overall_score:.4f}")

submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': preds
})
submission.to_csv('submission.csv', index=False)
print("Saved submission.csv!")

: 